# Stage 1: Candidate AML-Associated Protein Identification via PPI Network Analysis

This notebook runs the complete Stage 1 pipeline for identifying candidate AML-associated proteins by integrating gene-expression analysis with protein–protein interaction (PPI) network topology.

**Important**: All outputs are *candidate* AML-associated proteins — not experimentally validated AML drivers.

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), 'src'))
from config import *

## Step 1: Data Loading

In [ ]:
from data_loader import load_and_prepare_data
data = load_and_prepare_data()

## Steps 2-3: Differential Expression & Gene Selection

In [ ]:
from differential_expression import run_differential_expression
de_results, selected_genes = run_differential_expression(
    data['expression'], data['gene_meta'],
    data['aml_sample_ids'], data['all_sample_ids']
)

In [ ]:
from IPython.display import Image
Image(filename=os.path.join(RESULTS_DIR, 'volcano_plot.png'))

## Step 4: Gene → Protein Mapping

In [ ]:
from gene_protein_mapping import map_genes_to_proteins
mapping_df, mapped_proteins, id_to_name = map_genes_to_proteins(selected_genes)

## Steps 5-6: PPI Network Construction

In [ ]:
from ppi_network import build_ppi_network
G, edge_df, network_stats = build_ppi_network(mapped_proteins, id_to_name)

In [ ]:
Image(filename=os.path.join(RESULTS_DIR, 'ppi_network.png'))

## Steps 7-8: Centrality Analysis & Ranking

In [ ]:
from network_analysis import compute_centrality_and_rank
centrality_all, ranking_seeds = compute_centrality_and_rank(G)

In [ ]:
Image(filename=os.path.join(RESULTS_DIR, 'top_proteins_ranking.png'))

## Step 9: Permutation Test

In [ ]:
from permutation_test import run_permutation_test
n_seeds = (centrality_all['node_type'] == 'aml_seed').sum()
perm_results = run_permutation_test(centrality_all, n_seeds)

In [ ]:
Image(filename=os.path.join(RESULTS_DIR, 'permutation_null_distribution.png'))

## Step 10: Functional Enrichment

In [ ]:
from enrichment import run_enrichment
enrichment_df = run_enrichment(ranking_seeds, mapped_proteins, id_to_name)

In [ ]:
Image(filename=os.path.join(RESULTS_DIR, 'enrichment_plot.png'))

## Step 11: Independent Validation

In [ ]:
from validation import run_independent_validation
validation_df = run_independent_validation(selected_genes)